# 3장 1강: CNN의 구조와 원리 — 실습문제

## 실습 목표

- 작은 필터를 이동하며 곱하고 더하는 합성곱 계산을 이해합니다.
- 같은 입력에서도 필터에 따라 다른 특징 맵이 만들어지는 이유를 설명합니다.
- 풀링으로 특징 맵을 요약하고, 남는 정보와 사라지는 정보를 구분합니다.
- CNN 기반 이미지 분류의 처리 순서와 각 층의 역할을 설명합니다.

## 실습 환경 / 데이터

- Python, pandas, NumPy를 사용합니다. 별도의 딥러닝 프레임워크는 필요하지 않습니다.
- **`jena_climate_2009_2016.csv`를 노트북과 같은 폴더에 두고**, 해당 폴더를 VS Code에서 열어 실행하세요. pandas와 NumPy가 설치된 Python 환경을 커널로 선택합니다.
- 제공된 파일은 **420,551행, 15열**입니다. 이번에는 **첫 16행**을 읽고, 그중 `Date Time`과 기온 열 `T (degC)`를 사용합니다. 기온 단위는 섭씨(℃)입니다.
- 실습은 첫 8개 관측값, 과제는 다음 8개 관측값을 사용합니다. 선택한 구간의 관측 간격은 10분입니다.
- 코드 셀은 준비 셀부터 위에서 아래로 실행하세요. `np.round(..., 4)`는 출력만 소수 넷째 자리까지 표시하며, 계산은 원래 값으로 합니다.

## 이번 강의의 범위

제공된 **3장 1강 목차: Convolution·Filter·Feature Map, Pooling과 특징 추출, CNN 기반 이미지 분류 구조**를 기준으로 구성했습니다. 강의 본문은 별도로 제공되지 않았으므로 필터 값과 크기 예시는 개념 확인을 위해 정한 값입니다.

Jena Climate는 기상 시계열 데이터입니다. **시간 방향의 1차원 합성곱**으로 원리를 계산하고, 이미지의 **가로·세로 방향 2차원 합성곱**과 연결합니다. 연속 기온값을 임의로 이미지처럼 바꾸지는 않습니다. 이미지 분류는 구조와 역할을 묻는 개념 문제이며, 실제 이미지 모델을 학습하거나 분류 정확도를 측정하는 실습은 아닙니다.

필터는 계산을 확인하기 위해 고정해서 제공합니다. 실제 CNN에서는 일반적으로 필터의 가중치와 편향을 학습합니다. 이번에는 역전파 구현, 모델 학습, 미래 기온 예측, 성능 튜닝을 요구하지 않습니다. ReLU는 1장 2강에서 배운 내용을 그대로 사용합니다.

## 실습 구성

| 구분 | 문제 | 할 일 |
|---|---|---|
| 실습 1 | 1-1 | 두 필터로 특징 맵을 만들고 첫 값을 손계산하기 |
| 실습 2 | 2-1 | ReLU 후 Max Pooling·Average Pooling 비교하기 |
| 실습 2 | 2-2 | CNN 이미지 분류 구조와 각 단계의 크기 이해하기 |
| 기초 과제 | 3-1 | 다음 8개 기온에 같은 합성곱·풀링 적용하기 |

**실습 2개 + 기초 과제 1개**이며, 마지막에는 **실습 마무리 복습 질문 6개**가 있습니다. 과제는 실습에서 사용한 함수와 순서를 그대로 적용하는 수준입니다.

## 시작 전에 읽을 용어

| 용어 | 쉽게 풀어쓴 뜻 | 이번 실습의 예 |
|---|---|---|
| CNN | 주변 값들의 패턴을 필터로 추출하는 신경망 | 기온의 작은 구간을 차례로 관찰 |
| Convolution, 합성곱 | 작은 구간과 필터를 대응 위치끼리 곱한 뒤 더하는 계산 | 기온 3개와 필터 값 3개를 곱하고 합침 |
| Filter, 필터·커널 | 어떤 패턴에 반응할지 정하는 가중치 묶음 | `[−1, 0, 1]` |
| Feature Map, 특징 맵 | 필터를 여러 위치에 적용해 얻은 결과 배열 | 한 필터로 만든 숫자 6개 |
| Stride, 이동 간격 | 다음 계산 위치로 얼마나 이동하는지 | 합성곱 1칸, 풀링 2칸 |
| Padding, 패딩 | 입력 가장자리에 값을 덧붙이는 것 | 이번에는 덧붙이지 않음 |
| ReLU | 음수를 0으로, 양수를 그대로 통과시키는 함수 | 변화량 중 양의 반응을 남김 |
| Pooling, 풀링 | 작은 구역을 대표값 하나로 줄이는 계산 | 2개씩 묶어 하나만 남김 |
| Max Pooling | 묶음의 최댓값을 남기는 풀링 | `[0.24, 0.26]`에서 0.26 |
| Average Pooling | 묶음의 평균을 남기는 풀링 | `[0.24, 0.26]`에서 0.25 |
| Channel, 채널 | 같은 위치에 대응하는 특징의 종류 | 필터 2개가 만드는 특징 맵 2개 |
| Flatten | 여러 차원의 값을 한 줄로 펼치는 단계 | 3×3×2개 값을 길이 18로 펼침 |
| Dense, 완전연결층 | 추출한 특징들을 연결해 출력값을 계산하는 층 | 앞 강의의 가중합으로 클래스별 점수 계산 |

### 계산 규칙

이번 합성곱은 **필터 길이 3, stride 1, padding 없음, bias 0**입니다. 필터를 뒤집지 않고 적힌 순서대로 계산합니다. 딥러닝에서 합성곱이라고 부르는 연산은 보통 이 방식(엄밀히는 교차상관)을 사용합니다. 용어 차이를 암기할 필요는 없습니다.

$$f_i=x_i k_0+x_{i+1}k_1+x_{i+2}k_2+0$$

`x`는 입력 기온, `k`는 필터, `i`는 계산을 시작하는 위치입니다. 길이 8에 길이 3 필터를 한 칸씩 적용하면 **8−3+1=6개** 결과가 생깁니다. 이 간단한 길이 계산은 stride 1이고 padding이 없을 때 사용합니다.

`[−1, 0, 1]`의 결과는 **세 번째 기온−첫 번째 기온**입니다. 두 관측 시점은 20분 떨어져 있습니다. 이는 20분 동안의 기온 차이이며, 시간을 나눈 변화율이나 미래 예측값은 아닙니다.


## 실습 준비 — 제공 코드를 그대로 실행하세요

첫 셀은 필요한 데이터만 읽습니다. `iloc[:8]`은 처음 8행, `iloc[8:16]`은 9번째부터 16번째 행입니다. Python의 위치 번호는 0부터 시작하며 끝 번호는 포함하지 않습니다.

두 번째 셀의 함수는 계산을 도와주는 **제공 함수**입니다. 함수를 새로 작성하는 것이 문제는 아닙니다.

- `conv1d_valid(values, kernel)`: 필터가 들어갈 수 있는 모든 위치를 한 칸씩 이동합니다. 대응 위치를 곱한 뒤 `np.sum()`으로 합칩니다. 편향은 0입니다.
- `pool_pairs(values)`: 값을 두 개씩 겹치지 않게 묶어 최댓값 배열과 평균 배열을 순서대로 돌려줍니다. 이 수업에서는 길이가 짝수인 입력만 사용합니다.
- `max_values, avg_values = pool_pairs(...)`처럼 반환값 두 개를 각각 저장합니다.


### 코드 읽기 안내 — 처음 사용하는 기호부터 확인하기

| 표현 | 읽는 방법과 하는 일 |
|---|---|
| `# 설명` | 주석입니다. 사람이 읽는 설명이며 Python이 계산하지 않습니다. |
| `x = ...` | 오른쪽 결과를 계산한 뒤 `x`라는 이름으로 보관합니다. 수학의 등식 증명과는 다릅니다. |
| `[...]` | 여러 값을 순서대로 적거나, 배열에서 필요한 부분을 선택할 때 씁니다. |
| `np.array([값1, 값2, 값3])` | 여러 숫자를 계산 가능한 NumPy 배열로 만듭니다. 원소는 배열 안의 숫자 하나를 뜻합니다. |
| `함수(입력)` | 미리 정해 둔 처리 과정에 입력을 넣어 실행합니다. `print(x)`는 x를 화면에 표시합니다. |
| `len(x)` | x에 들어 있는 원소 수를 셉니다. 값들의 합을 구하는 함수가 아닙니다. |
| `x[a:b]` | 위치 a부터 b 바로 전까지 선택합니다. Python의 첫 위치 번호는 0입니다. |
| `def 함수이름(...):` | 다음에 사용할 처리 방법을 정의합니다. 정의만으로 입력 데이터의 계산이 끝나는 것은 아닙니다. |
| `for ...:` | 정해진 위치를 차례로 바꾸며 같은 일을 반복합니다. 안으로 들여쓴 줄들이 반복됩니다. |
| `return 결과` | 함수가 계산한 결과를 호출한 곳으로 돌려줍니다. 화면에 출력하는 `print`와 역할이 다릅니다. |

**실행 순서:** 데이터 읽기 → 제공 함수 정의 → 실습 1 → 실습 2 → 과제입니다. 셀은 노트북에서 한 번에 실행하는 상자입니다. 앞 셀에서 만든 변수와 함수는 뒤 셀에서 사용합니다. 실행 순서를 건너뛰면 `x`나 `conv1d_valid`를 찾지 못할 수 있습니다.

**함수의 입력과 출력:** `conv1d_valid(x, k_mean)`에서 x는 함수 안의 `values`로, k_mean은 `kernel`로 전달됩니다. 함수가 돌려준 배열을 `f_mean = ...`으로 받습니다. `max_map, avg_map = pool_pairs(activated)`는 함수가 돌려준 두 배열을 왼쪽 이름에 순서대로 저장합니다. 두 이름이 있으므로 두 번 실행하는 것이 아닙니다.

`pool_pairs`는 이번 수업에서 사용하는 **짝수 길이의 1차원 배열**을 대상으로 합니다. 모든 크기의 입력이나 이미지에 그대로 적용하는 범용 CNN 함수는 아닙니다.


In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv("jena_climate_2009_2016.csv", nrows=16)
times = pd.to_datetime(df["Date Time"], format="%d.%m.%Y %H:%M:%S")
# 위치 0~7의 기온 8개를 실습용 숫자 배열로 만듭니다. 끝 번호 8은 포함하지 않습니다.
x = df["T (degC)"].iloc[:8].to_numpy(dtype=float)
# 위치 8~15의 기온 8개를 과제용 배열로 만듭니다. 끝 번호 16은 포함하지 않습니다.
x_home = df["T (degC)"].iloc[8:16].to_numpy(dtype=float)

# 행·열 개수와 선택한 날짜·기온을 화면에 표시합니다.
print("읽은 행과 열:", df.shape)
print(df[["Date Time", "T (degC)"]].to_string(index=False))
# 이웃한 시각 차이를 초 단위로 바꾼 뒤 60으로 나누어 분 단위로 확인합니다.
print("선택 구간 간격(분):", times.diff().dropna().dt.total_seconds().unique() / 60)
# 기온이 비어 있는 칸의 개수를 확인합니다.
print("선택 기온의 결측값 수:", df["T (degC)"].isna().sum())
print("실습 입력:", x)
print("과제 입력:", x_home)


읽은 행과 열: (16, 15)
          Date Time  T (degC)
01.01.2009 00:10:00     -8.02
01.01.2009 00:20:00     -8.41
01.01.2009 00:30:00     -8.51
01.01.2009 00:40:00     -8.31
01.01.2009 00:50:00     -8.27
01.01.2009 01:00:00     -8.05
01.01.2009 01:10:00     -7.62
01.01.2009 01:20:00     -7.62
01.01.2009 01:30:00     -7.91
01.01.2009 01:40:00     -8.43
01.01.2009 01:50:00     -8.76
01.01.2009 02:00:00     -8.88
01.01.2009 02:10:00     -8.85
01.01.2009 02:20:00     -8.83
01.01.2009 02:30:00     -8.66
01.01.2009 02:40:00     -8.66
선택 구간 간격(분): [10.]
선택 기온의 결측값 수: 0
실습 입력: [-8.02 -8.41 -8.51 -8.31 -8.27 -8.05 -7.62 -7.62]
과제 입력: [-7.91 -8.43 -8.76 -8.88 -8.85 -8.83 -8.66 -8.66]


In [7]:
# 함수 정의: 입력 배열 values와 필터 배열 kernel을 받아 특징 맵을 돌려줍니다.
def conv1d_valid(values, kernel):
    results = []  # 계산 결과를 차례로 모을 빈 목록입니다.
    # 길이 8의 입력과 길이 3의 필터라면 시작 위치 i는 0, 1, 2, 3, 4, 5입니다.
    for i in range(len(values) - len(kernel) + 1):
        window = values[i:i + len(kernel)]  # 현재 위치부터 필터 길이만큼 꺼냅니다.
        value = np.sum(window * kernel)  # 같은 위치끼리 곱하고, 그 결과를 모두 더합니다.
        results.append(value)  # 현재 구간의 결과 한 개를 목록 맨 뒤에 붙입니다.
    return np.array(results)  # 반복이 끝난 뒤 모든 결과를 NumPy 배열로 돌려줍니다.


# 값을 두 개씩 묶어 최댓값 배열과 평균 배열을 함께 돌려주는 함수입니다.
def pool_pairs(values):
    maximums = []  # 각 묶음의 최댓값을 모읍니다.
    averages = []  # 각 묶음의 평균을 모읍니다.
    for i in range(0, len(values), 2):  # 길이 6일 때 i는 0, 2, 4입니다.
        window = values[i:i + 2]  # 현재 위치부터 두 개를 꺼냅니다.
        maximums.append(np.max(window))  # 두 값 중 큰 값 하나를 저장합니다.
        averages.append(np.mean(window))  # 두 값을 더하고 2로 나눈 값을 저장합니다.
    return np.array(maximums), np.array(averages)  # 최댓값이 첫째, 평균이 둘째입니다.


print("합성곱 함수와 풀링 함수 준비 완료")


합성곱 함수와 풀링 함수 준비 완료


---

## 실습 1. Convolution과 특징 추출


### 문제 1-1. 같은 기온에 두 가지 필터 적용하기

#### 문제 설명

첫 8개 기온 `x`에 다음 필터를 각각 적용하세요. 두 필터 모두 같은 입력에서 독립적으로 출발합니다.

| 필터 | 값 | 살펴볼 패턴 |
|---|---|---|
| 평균 필터 `k_mean` | `[1/3, 1/3, 1/3]` | 주변 3개 기온의 평균 수준 |
| 차이 필터 `k_diff` | `[−1, 0, 1]` | 구간 끝 기온과 시작 기온의 차이 |

#### 요구사항

1. `np.array()`로 두 필터를 만드세요.
2. 제공 함수로 `f_mean`, `f_diff`를 구하고 두 특징 맵과 길이를 출력하세요.
3. 첫 구간 `x[:3]`에 각 필터를 곱한 값과 그 합을 손계산하세요.
4. 차이 특징 맵의 부호를 보고 각 구간에서 끝 기온이 시작보다 높은지 낮은지 설명하세요.

#### 작성 도움말

`conv1d_valid(x, k_mean)`처럼 입력과 필터를 넣습니다. `len()`은 값의 개수를 반환합니다. 두 번째 필터도 `x`에 적용해야 하며 첫 특징 맵에 이어서 적용하지 않습니다.

#### 해석 질문

**Q1.** 입력 기온, 필터, 특징 맵은 각각 무엇인가요? 길이 8인 입력의 특징 맵 길이가 6인 이유는 무엇인가요?

**Q2.** 두 필터의 첫 번째 출력값은 각각 얼마인가요? 곱셈과 덧셈 과정을 쓰세요.

**Q3.** 차이 특징 맵에서 음수와 양수는 무엇을 뜻하나요? 양수라면 세 관측값이 모두 연속해서 증가했다는 뜻인가요?

**Q4.** 같은 입력에 다른 필터를 적용하면 왜 결과가 달라지나요? 이번 필터와 실제 CNN에서 학습하는 필터의 차이도 설명하세요.

#### 제출 결과

두 특징 맵·길이, 첫 출력값 손계산, Q1~Q4 답변.


In [8]:
# 두 필터를 np.array()로 만드세요.
# k_mean = ...
# k_diff = ...

# 같은 x에 각 필터를 적용하세요.
# f_mean = ...
# f_diff = ...

# 두 특징 맵과 각각의 길이를 출력하세요.

# 세 값을 같은 비중으로 평균 낼 필터
k_mean = np.array([1/3, 1/3, 1/3]) # 세 기온을 같은 비중으로 반영했기 때문에 1/3을 값으로 사용했고

# 마지막 기온에서 처음 기온을 뺄 필터
k_diff = np.array([-1.0, 0.0, 1.0]) # 첫 기온은 빼고 마지막 기온은 더하는 구조(-1, 0, 1 계산 규칙을 정하는 가중치 부여)

# 원래 기온 x에서 평균 특징을 추출
f_mean = conv1d_valid(x, k_mean) # x에 평균 필터를 이동시키면서 적용, 주변에 3개의 기온, 즉, 평균 6개가 f_mean에 저장

# f_mean이 아니라 같은 원래 기온 x에서 차이 특징을 추출
f_diff = conv1d_valid(x, k_diff) # 같은 x에 차이 필터를 따로 적용, 끝 - 시작의 기온 차이 6개가 f_diff에 저장

#출력문
print("평균 특징 맵:", np.round(f_mean,4))
print("차이 특징 맵:", np.round(f_diff,4))
print("특징 맵 길이:", len(f_mean), len(f_diff))
print("첫 구간:", x[:3])
print("평균 필터 곱셈:", np.round(x[:3] * k_mean,4))
print("차이 필터 곱셈:", np.round(x[:3] * k_diff,4))

평균 특징 맵: [-8.3133 -8.41   -8.3633 -8.21   -7.98   -7.7633]
차이 특징 맵: [-0.49  0.1   0.24  0.26  0.65  0.43]
특징 맵 길이: 6 6
첫 구간: [-8.02 -8.41 -8.51]
평균 필터 곱셈: [-2.6733 -2.8033 -2.8367]
차이 필터 곱셈: [ 8.02 -0.   -8.51]


#### 답변 작성란

**수식 및 계산 과정**

- 필터가 이동하는 모든 위치 값 찍어보기

|시작위치|선택구간|입력 기온 3개|차이계산 : 마지막 - 처음|
|---|---|---|---|
|0|x[0:3]|[-8.02, -8.41, -8.51]|-8.51 -(-8.02) = -0.49|
|1|x[1:4]|[-8.41, -8.51, -8.31]|-8.31 - (-8.41) = 0.10|
|2|x[2:5]|[-8.51, -8.31, -8.27]|-8.27-(-8.51)=0.24|
|3|x[3:6]|[-8.31, -8.27, -8.05]|-8.05 -(-8.31) = 0.26|
|4|x[4:7]|[-8.27, -8.05, -7.62]|-7.62 -(-8.27) = +0.65|
|5|x[5:8]|[-8.05, -7.62, -7.62]|-7.62 -(-8.05) = +0.43|

- 구간을 한 칸씩 옮기겠다고 설정했으니까 이웃한 구간 입력 2개 값을 공유하여 계산한다.
- 새롭게 칸을 옮길때마다 가중치 숫자는 그대로고 입력구간은 달라진다.
- 입력 위치 6부터 시작하면 두 값만 남기 때문에 더 계산하지 않는다.
- 지금 0~6까지 6개의 결과라고 적힌 이유다.

**Q1. 입력 기온, 필터, 특징 맵은 각각 무엇인가요? 길이 8인 입력의 특징 맵 길이가 6인 이유는 무엇인가요?**

- 답변: 입력은 시간 순서대로 배열한 기온 8개
- 필터는 세 관측값을 어떤 비중으로 결합할 지 정하는 숫자 3개
- 특징 맵은 그 필터를 이동하며 얻은 결과
- 시작 위치가 0~5까지 이니까 8-3+1 = 6개
- 위치 6개에서 시작하면 입력이 2개밖에 없으니까 3필터를 적용할 수 없다.

**Q2. 두 필터의 첫 번째 출력값은 각각 얼마인가요? 곱셈과 덧셈 과정을 쓰세요.**

- 답변: 평균 필터는 (-8.02)/3 + (-8.41)/3 + (-8.51)/3 = -24.94/3 = -8.3133
- 차이 필터는 (-8.02) x (-1) + (-8.41) x 0 +(-8.51) x 1 = 8.02 + 0 -8.51 = -0.49
- 기온은 가중치가 0으로 차이 게산에 기여하지 않는다.

**Q3. 차이 특징 맵에서 음수와 양수는 무엇을 뜻하나요? 양수라면 세 관측값이 모두 연속해서 증가했다는 뜻인가요?**

- 답변: 음수는 20분 뒤의 구간 끝 기온이 시작보다 낮다라는 의미,양수는 높다라는 의미
- 양수여도 중간기온까지 계속 증가했다고 해석할 수 없음
- 예를 들어 [-8.41, -8.51, -8.31]은 한번 내려간 뒤 올라지만 끝 - 시작 = 0.10이다.

**Q4. 같은 입력에 다른 필터를 적용하면 왜 결과가 달라지나요? 이번 필터와 실제 CNN에서 학습하는 필터의 차이도 설명하세요.**

- 답변: 필터가 각 위치에 주는 가중치가 다르기 때문이다.
- 평균 필터는 세 값에 같은 양의 비중을 주고
- 차이 필터는 처음 값을 빼고 마지막 값을 더한다.
- 이번 예제에서는 CNN의 원리를 쉽게 해석하기 위해 고정한 필터를 사용했고
- 실제 CNN에서는 보통 학습 데이터와 손실을 이용하여 필터의 가중치와 편향을 업데이트하여 필요한 특징을 추출한다.


---

## 실습 2. Pooling과 CNN 이미지 분류 구조


### 문제 2-1. 특징 맵을 ReLU와 Pooling으로 요약하기

#### 문제 설명

문제 1-1의 **차이 특징 맵 `f_diff`만** 사용합니다. 먼저 ReLU를 적용하고, 그 결과를 두 개씩 묶어 Max Pooling과 Average Pooling을 각각 수행하세요.

$$a_i=max(0,f_i)$$

두 값 `[u, v]`를 묶었을 때 Max Pooling은 `max(u, v)`, Average Pooling은 `(u+v)/2`입니다. **풀링 크기 2, stride 2, padding 없음**으로 고정합니다.

#### 요구사항

1. `np.maximum(0.0, f_diff)`로 ReLU 결과 `activated`를 만드세요.
2. `pool_pairs(activated)`로 `max_map`, `avg_map`을 받으세요. 두 풀링은 같은 activated에서 각각 출발합니다.
3. 합성곱 결과, ReLU 결과, 두 풀링 결과와 길이를 출력하세요.
4. 세 묶음을 직접 적고 각 묶음의 최댓값과 평균을 계산하세요.

#### 해석 질문

**Q1.** ReLU는 이번 차이 특징 맵에서 어떤 정보를 남기고 어떤 정보를 없애나요?

**Q2.** Max Pooling과 Average Pooling의 세 결과는 각각 얼마인가요? 길이는 왜 6에서 3이 되나요?

**Q3.** 합성곱 필터와 이 두 풀링의 역할은 어떻게 다른가요? 일반적인 Max/Average Pooling에 학습할 가중치가 있나요?

**Q4.** 풀링 결과만 보고 입력 특징 맵을 정확히 복원할 수 있나요? 값의 개수를 줄이면 항상 분류가 더 잘되나요?

#### 제출 결과

단계별 결과·길이, 세 묶음의 풀링 계산, Q1~Q4 답변.


In [9]:
# f_diff에 ReLU를 적용하세요.
# activated = ...

# 제공 함수로 두 풀링 결과를 받으세요.
# max_map, avg_map = ...

# 각 단계의 결과와 길이를 출력하세요.

# 앞서 구한 f_diff의 각 값과 0 중 큰 값을 남긴다. 길이는 바꾸지 않음
activated = np.maximum(0.0, f_diff) # ReLU처럼 -값을 0으로 바꿈, -0.49 -> 0으로

# 반환 순서에 맞춰 최대값 배열과 평균 배열을 각각 받는다.
max_map, avg_map = pool_pairs(activated) # ReLU 결과를 두개씩 묶어서 최대값, 평균 계산
                                         # 첫 번째 배열은 max_map, 두번째 반환 배열은 avg_map에 저장

# 같은 입력이 각 자리 단계를 거치며 어떻게 달라지는지 비교합니다.
print("합성곱 결과:", np.round(f_diff,4), "길이:", len(f_diff)) # 필터로 추출한 원래 차이 특징 6개
print("ReLU 결과:", np.round(activated,4), "길이:", len(activated)) # 값 일부는 바뀌지만 위치는 6개
print("최대 풀링 결과:", np.round(max_map,4), "길이:", len(max_map)) # 각 묶음에서 1개씩 남기 때문에 3개
print("평균 풀링 결과:", np.round(avg_map,4), "길이:", len(avg_map)) # 최대값과 별개로 묶은 평균 3개


합성곱 결과: [-0.49  0.1   0.24  0.26  0.65  0.43] 길이: 6
ReLU 결과: [0.   0.1  0.24 0.26 0.65 0.43] 길이: 6
최대 풀링 결과: [0.1  0.26 0.65] 길이: 3
평균 풀링 결과: [0.05 0.25 0.54] 길이: 3


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

|함수 안 시작 위치|선택 위치|두 값|Max 계산|Average 계산|
|---|---|---|---|---|
|0|0,1|[0.00, 0.10]|두 수 중 큰 0.10을 선택|(0.00+0.10)/2 = 0.05|
|2|2,3|[0.24, 0.26]|두 수 중 큰 0.26을 선택|(0.24+0.26)/2 = 0.25|
|4|4,5|[0.65, 0.43]|두 수 중 큰 0.65을 선택|(0.65+0.43)/2 = 0.54|

- 여기서 크기 2는 한번에 읽는 값이 두 개라는 뜻
- stride 2는 다음 묶음의 시작점을 두 칸 옮기겠다는 뜻
- 묶음이 3개인데 6개를 2개씩 묶어서
- 최종적으로 각 묶음이 숫자 하나를 만들기 위해 출력 길이가 3
- Max Pooling은 각 묶음별로 더 강한 양의 반응에 민감하게 반응
- Average Pooling은 묶음의 양의 반응을 평균하여 계산
- 폴링 결과는 원리 개온의 최대값의 평균이 아니라 ReLU를 통과한 차이 특징의 요약 결과
- 마지막 0.65는 0.65도 만큼 차이가 난다는 뜻이나 0.65도가 아님

**Q1. ReLU는 이번 차이 특징 맵에서 어떤 정보를 남기고 어떤 정보를 없애나요?**

- 답변: 음의 차이인 -0.49 값을 0으로 만들어서 기온이 낮아진 정도를 없애고 양의 차이만을 남겼다.
- 여기서 0이라는 것은 원래 차이가 0이였거나 음수여서 잘린 경우를 모두 포함한다.
- ReLU가 모든 정보를 보존하는 것은 아니기 때문에 원래 기온이나 특징맵에서는 ReLU를 무조건 적으로 사용할 수는 없다.

**Q2. Max Pooling과 Average Pooling의 세 결과는 각각 얼마인가요? 길이는 왜 6에서 3이 되나요?**

- 답변: Max는 [0.10, 0.26, 0.65], average는 [0.05, 0.25, 0.54]이다.
- 두 개를 하나로 줄이는 겹치지 않는 묶음이 세 개라서 길이가 3개다.

**Q3. 합성곱 필터와 이 두 풀링의 역할은 어떻게 다른가요? 일반적인 Max/Average Pooling에 학습할 가중치가 있나요?**

- 답변: 합성곱은 필터의 가중치로 지역적인 패턴을 계산한다.
- 폴링은 이미 만들어진 특징을 정해진 규칙으로 요약한다.
- 일반적으로 max / average pooling 자체에는 학습할 가중치나 편향이 없다.
- 폴링이나 이동 간격은 설계자가 직접 정하는 설정 값이다.

**Q4. 풀링 결과만 보고 입력 특징 맵을 정확히 복원할 수 있나요? 값의 개수를 줄이면 항상 분류가 더 잘되나요?**

- 답변: 정확한 복원은 일반적으로 불가능하다.
- 최댓값 0.26만 보고 나머지 값이 0.24였는지 0.10이었는지는 알 수 없다.
- 평균 하나라도 원래 두 값을 유일하게 정할 수 없다.
- 작은 특징맵을 가진다는 것은 계산량을 줄이는 데에는 도움이 되지만 세부 정보까지도 줄어든다.
- 따라서 폴링 자체가 언제나 분류 성능을 높인다고 결론을 내릴수는 없다.

### 문제 2-2. CNN 기반 이미지 분류의 처리 흐름 연결하기

#### 문제 설명

이번 문제는 **코드 없이 설명하는 구조 문제**입니다. 기온 배열은 시간 순서의 1차원 입력이고 이미지는 높이와 너비가 있는 2차원 입력입니다. 필터를 이동하며 주변 특징을 찾는 원리는 두 경우에 공통입니다.

다음은 이미지 1장을 세 종류 중 하나로 분류하는 **설명용 CNN 구조**입니다. 실제 이미지 파일이나 학습 결과는 사용하지 않습니다.

- 입력: 높이 8, 너비 8, 채널 1인 흑백 이미지.
- 합성곱: 3×3 필터 2개, stride 1, padding 없음. 각 필터는 입력의 채널 전체를 읽습니다.
- 활성화: ReLU.
- 풀링: 각 특징 맵에 2×2 Max Pooling, stride 2, padding 없음.
- Flatten: 모든 특징 값을 순서대로 한 줄로 펼침.
- Dense: 클래스 A·B·C의 점수 3개를 출력.

이번 표는 이미지 **한 장**을 나타내므로 여러 장을 묶는 배치 차원은 생략합니다. 배열 표기 순서는 **높이×너비×채널**로 통일합니다.

#### 요구사항

1. 다음 항목을 올바른 순서로 배열하세요: `Dense → 입력 이미지 → Pooling → Convolution → Flatten → ReLU`.
2. 입력, 합성곱, ReLU, 풀링, Flatten, Dense의 출력 크기를 쓰세요.
3. 각 단계가 하는 일을 한 문장씩 설명하세요.
4. 설명용 출력 점수가 **[A: 0.2, B: 1.4, C: −0.3]**일 때 가장 높은 점수의 클래스를 고르세요. 이 숫자는 확률로 변환하지 않은 가상의 점수입니다.

#### 작성 도움말

합성곱 결과의 높이와 너비는 각각 `8−3+1`입니다. 필터 한 개가 특징 맵 하나를 만들므로 필터 2개면 출력 채널이 2개입니다. ReLU는 크기를 바꾸지 않습니다. 6×6 특징 맵을 2×2씩 겹치지 않게 묶으면 3×3이 됩니다. Flatten은 값의 개수를 유지합니다.

#### 해석 질문

**Q1.** 올바른 처리 순서와 각 단계의 출력 크기는 무엇인가요?

**Q2.** 합성곱·ReLU·풀링·Flatten·Dense는 각각 어떤 역할을 하나요?

**Q3.** 예시 점수에서 선택할 클래스는 무엇인가요? B의 점수 1.4를 140%의 확률로 해석해도 되나요?

**Q4.** Jena 기온 실습과 이미지 분류 구조는 무엇이 같고 무엇이 다른가요? 기온 특징 맵을 계산한 것만으로 이미지 분류 모델 학습까지 완료했다고 할 수 있나요?

#### 제출 결과

단계별 순서·크기·역할 표, 클래스 선택, Q1~Q4 답변.


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 1단계 : 합성곱에서 8x8이 6x6이 되는 이유
- 3x3 필터는 한번에 이웃한 밝기값 9개를 읽는다. 각 값과 같은 위치의 필터 가중치를 곱하고 9개의 결과를 더한다
- 기온 실습에서 세 값을 곱해 더했던 것과 같은 원리이지만 가로 x 세로가 있는 구역
- 가로 8칸에 너비 3인 필터를 올리면 왼쪽의 시작 위치는 0,1,2,3,4,5까지 가능
- 6부터 시작하면 6번, 7번 칸이 2개가 남아서 너비 3을 채우지 못한다.
- 따라서 가로 결과가 6개고, 세로 결과도 마찬가지로 6개다.
- 필터 1개가 6 x 6 = 36개의 값의 특징 맵 하나를 만든다.
- 필터가 2개면 같은 이미지를 두 기준으로 읽어 특징 맵이 2장이 생긴다.
- 두장을 더해 한장으로 만들지는 않기 때문에 6 x 6 x 2
- 3 x 3은 한 필터의 공간 크기
- 2개는 핉터 수

- 2단계 ReLU는 값만 바꾸고 배열 크기는 유지
- 각 특징 맵의 음수 값을 0으로 바꾼다.
- 0으로 바뀐 자리도 남아있기 때문에 결과의 크기는 6 x 6 x 2
- ReLU가 비선형이라는 말은 입력에 같은 가중치를 곱해 더하는 계산만으로 나타내기 어려운 변환을 한다는 뜻

- 3단계, 2x2 max pooling은 채널마다 따로 계산한다.
- 6 x 6 특징 맵이 생성 된 후 2행 2열, 즉 4개의 값을 묶어서 가장 큰 값 1개를 남기겠다.
- 가로로 3묶음, 세로로 3묶음이 있으면 3x3
- 두 특징맵을 각각 처리하기 때문에 채널수는 여전히 2이며
출력크기는 3 x 3 x 2로 나타난다.
- 기온 코드의 폴링은 숫자 2개를 묶는 방식으로 계산했지만 이미지의 경우
- 2x2 폴링으로 숫자 4개를 묶는다.
- 둘 다 주변의 여러 특징을 가지고 대표값 1개만 남긴다는 원리는 같다.

- 4단게 Flatten은 18개 값을 그대로 한줄로 펼친다.
- 폴링 결과에는 3x3 크기의 특징 맵이 2개가 있다고 했으니까 3x3x2=18개를 의미한다.
- Flatten은 18개의 값을 정해진 순서대로 나열한다.
- 합계를 하거나 곱셈을 만들거나 평균을 구하지 않기 때문에 값 개수는 18개로 변하지 않는다.
- 최종 모든 값의 길이를 18의 배열로 연결한다는 뜻으로 해석할 수 있다.

- Dense는 18갣의 특징을 세개의 점수로 연결한다.
- Dense는 1장에서 배운 가중치를 곱해 더하고 편향을 더하는 게산을 의미하는데
- a점수를 만들 때 18개의 특징이 각각 a용 가중치를 곱해 합하고 a용 편향을 더한다.
- b와 c도 각각 가중치와 편향으로 계산한다. 따라서 입력 특징 자체는 18개지만 출력 점수는 클래스의 수인 3개다.
예시로 [0.2, 1.4, -0.3]에서는 b의 1.4가 가장 큰 값이다. 그렇기 때문에 b를 선택한다.
- 주어진 점수에서 어느 클래스를 고르는지 확인한다.

-먼저 특징을 추출하여 그 특징으로 틀래스를 구분하는 부분을 연결한다.
- 앞쪽의 합성곱, ReLU, 폴링은 국소 특징을 처리하고 뒤쪽의 Flatten, Dense 부분은 이를 분류 출력부분으로 연결한다.
- 이 구조는 기초 예시 중 하나, CNN이 반드시 이 구조와 층수를 가지는 것은 아니다.

|순서|단께|출력크기|역할과 크기 계산|
|---|---|---|---|
|1|입력 이미지|8x8x1|8행 8열의 흑백 픽셀 이미지 값을 받음|
|2|Convolution|6x6x2|각 필터가 작은 구역의 특징 계산, 한 방향의 크기 8-3+1=6, 필터 2개로 채널 2개|
|3|ReLU|6x6x2|각 값의 음수를 0으로 바꿈, 위치와 채널 수는 유지|
|4|Max Pooling|3x3x2|채널마다 2x2 구역의 최댓값을 남김, 높이 너비는 6/2 = 3, 채널은 유지|
|5|Flatten|18|3x3x2=18개의 값을 한 줄로 펼침|
|6|Dense|3|펼친 특징을 가중치 편향으로 결합하여 A, B, C 점수 3개 출력|

- 합성곱 필터 2개는 같은 이미지를 읽지만 서로 다른 가중치를 가질 수 있다. <-- 즉 다른 특징을 추출 가능하다.
- 필터 크기 3x3은 한번에 볼 공간을 의미하며 필터 개수는 2개는 출력 특징의 종류의 수를 의미한다.
- 츨링은 높이와 너비 방향으로 각 채널을 따로 요약하며, 채널 2개를 따로 합치거나 하지 않는다.
- Flatten도 값을 더하거나 평균내지 않고 배열의 모양만 바꾼다.


**Q1. 올바른 처리 순서와 각 단계의 출력 크기는 무엇인가요?**

- 답변: 
- 입력 이미지 -> Convalution -> Relu -> Pooling -> Flatten -> Dense
- 크기는 차례대로 8x8x1 -> 6x6x2 -> 6x6x2 -> 3x3x2 -> 18 -> 3

**Q2. 합성곱·ReLU·풀링·Flatten·Dense는 각각 어떤 역할을 하나요?**

- 답변: 
- 합성곱은 작은 구역에서의 패턴 파악
- ReLU는 음수를 0으로 바꾸는 비선형 변환
- 플링은 주변 특징을 대표값으로 요약
- Flatten은 특징을 한 줄로 펼침
- Dense는 특징들을 결합해서 클래스별로 점수 계산

**Q3. 예시 점수에서 선택할 클래스는 무엇인가요? B의 점수 1.4를 140%의 확률로 해석해도 되나요?**

- 답변: 
- 가장 높은 점수 1.4의 B를 선택
- 주어진 숫자는 점수 1.4이고, 확률값이 아니기 때문에 140%로 해석하지 않는다.

**Q4. Jena 기온 실습과 이미지 분류 구조는 무엇이 같고 무엇이 다른가요? 기온 특징 맵을 계산한 것만으로 이미지 분류 모델 학습까지 완료했다고 할 수 있나요?**

- 답변:
- 두 경우 모두 작은 구간에 같은 필터를 반복하여 적용해 특징 맵을 만들고 플링으로 특징을 요약한다는 점은 공통점
- 기온은 시간 방향으로 필터가 이동하며 이미지는 주어진 높이, 너비 방향으로 이동한다.
- 기온 같은 경우 사람이 정한 필터만큼의 계산을 의미하기 때문에, 이미지의 입력, 정답 클래스 학습 과정이 없기 때문에 이미지 분류와는 차이가 있다.


---

## 기초 과제. 다른 기온 구간에 같은 계산 적용하기


### 문제 3-1. 합성곱부터 Max Pooling까지 이어서 계산하기

#### 문제 설명

준비 셀의 `x_home`은 9번째부터 16번째 관측값입니다. 실습과 동일하게 **차이 필터 `[−1, 0, 1]` → ReLU → Max Pooling** 순서로 계산하세요. 새 함수 작성이나 모델 학습은 필요하지 않습니다.

합성곱은 필터 길이 3, stride 1, padding 없음, bias 0입니다. 풀링은 크기 2, stride 2, padding 없음입니다.

#### 요구사항

1. `home_filter`를 만들고 `conv1d_valid(x_home, home_filter)`로 `home_feature`를 구하세요.
2. ReLU 결과 `home_relu`를 구하세요.
3. `pool_pairs(home_relu)`에서 첫 반환값을 `home_max`로 받으세요. 두 번째 반환값은 받아 두되 이번 과제에서 출력·해석하지 않아도 됩니다.
4. 입력, 합성곱, ReLU, Max Pooling의 값과 길이를 출력하세요.
5. 첫 합성곱 값과 세 풀링 결과를 손계산하세요.

#### 해석 질문

**Q1.** 첫 구간의 합성곱 결과는 얼마이며, 어떻게 계산하나요?

**Q2.** ReLU를 적용한 특징 맵과 Max Pooling 결과는 각각 무엇인가요? 0인 풀링 결과는 기온이 0℃라는 뜻인가요?

**Q3.** 네 단계의 길이는 어떻게 바뀌나요? 앞선 실습과 결과 숫자가 다른 이유는 무엇인가요?

**Q4.** 이미지 분류 구조에서는 이렇게 추출·요약한 특징을 어떤 단계로 연결해 클래스별 점수를 만드나요?

#### 제출 결과

단계별 배열·길이, 손계산, Q1~Q4 답변. 예측 성능 평가나 추가 모델 비교는 요구하지 않습니다.


In [10]:
# 실습에서 사용한 코드의 입력을 x_home으로 바꿔 적용하세요.
# home_filter = ...
# home_feature = ...
# home_relu = ...
# home_max, home_avg_unused = ...

# 네 단계의 값과 길이를 출력하세요.

# 1. 차이 필터 만들기
home_filter = np.array([-1.0, 0.0, 1.0])

# 2. 합성곱 계산
home_feature = conv1d_valid(x_home, home_filter)

# 3. ReLU 적용
home_relu = np.maximum(0.0, home_feature)

# 4. Max Pooling 적용
home_max, home_avg_unused = pool_pairs(home_relu)

# 5. 결과 및 길이 출력
print("입력:", x_home, "길이:", len(x_home))
print("합성곱:", np.round(home_feature, 4),
      "길이:", len(home_feature))
print("ReLU:", np.round(home_relu, 4),
      "길이:", len(home_relu))
print("Max Pooling:", np.round(home_max, 4),
      "길이:", len(home_max))

# 6. 첫 합성곱과 첫 풀링 계산 확인
print("첫 합성곱 구간:", x_home[:3])
print("첫 필터 곱셈:", np.round(x_home[:3] * home_filter, 4))
print("첫 합성곱 결과:", round(home_feature[0], 4))

print("첫 풀링 구간:", np.round(home_relu[:2], 4))
print("첫 Max Pooling 결과:", round(home_max[0], 4))



입력: [-7.91 -8.43 -8.76 -8.88 -8.85 -8.83 -8.66 -8.66] 길이: 8
합성곱: [-0.85 -0.45 -0.09  0.05  0.19  0.17] 길이: 6
ReLU: [0.   0.   0.   0.05 0.19 0.17] 길이: 6
Max Pooling: [0.   0.05 0.19] 길이: 3
첫 합성곱 구간: [-7.91 -8.43 -8.76]
첫 필터 곱셈: [ 7.91 -0.   -8.76]
첫 합성곱 결과: -0.85
첫 풀링 구간: [0. 0.]
첫 Max Pooling 결과: 0.0


|순서|필터가 적용되는 입력 3개|결과(합성곱)|
|---|---|---|
|1|-7.91 -8.43 -8.76|-0.85|
|2|-8.43 -8.76 -8.88|-0.45|
|3|-8.76 -8.88 -8.85|-0.09|
|4|-8.88 -8.85 -8.83|0.05|
|5|-8.85 -8.83 -8.66|0.19|
|6|-8.83 -8.66 -8.66|0.17|

- 입력 길이 : 8개
- 필터 길이 [-1, 0, 1] 3개
- 패딩 : 없음 0개
- 이동간격(strid) :1
- 합성곱 결과 : (8+2(0)-3/1)+1 = 5+1 = 6

#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. 첫 구간의 합성곱 결과는 얼마이며, 어떻게 계산하나요?**

- 답변: 첫 구간의 합성곱은  [-7.91, -8.43, -8.76], 필터는 [-1, 0, 1]로 같은 위치끼리 곱하고 더하면
- -7.91 x -1 + (-8.43) x 0 + (-8.76) x 1 = -0.85
- 첫 합성곱 결과는 -0.85

**Q2. ReLU를 적용한 특징 맵과 Max Pooling 결과는 각각 무엇인가요? 0인 풀링 결과는 기온이 0℃라는 뜻인가요?**

- 답변: ReLU는 음수를 0으로 바꾸고, Max Pooling은 두 값 중 최댓값을 선택한다.
- ReLU 결과는 [0.   0.   0.   0.05 0.19 0.17]이고  [0, 0], [0, 0.05], [0.19, 0.17] 두 값을 묶고 이 중에서 최댓값을 고르면 Max Pooling의 결과는 [0, 0.05, 0.19]이다.
- 아니요. 풀링 결과가 0인 것은 기온이 0℃라는 뜻이 아니고
- ReLU 이후 선택한 두 값 중 최댓값을 선택하는데 그 선택한 값이 0이라는 뜻이다.

**Q3. 네 단계의 길이는 어떻게 바뀌나요? 앞선 실습과 결과 숫자가 다른 이유는 무엇인가요?**

- 답변: 입력은 [-7.91 -8.43 -8.76 -8.88 -8.85 -8.83 -8.66 -8.66]로 길이가 8,
- 합성곱 결과는 필터 길이 3, Stride 1, Padding 없음이므로 8 - 3 + 1 = 6이다.
- ReLU는 음수인 값을 0으로 바꾸기 때문에 길이는 6을 유지한다.
- MaxPooling은 두개의 값이 묶이기 때문에 길이가 3이다.
- 이전 실습과 같은 필터와 계산방식을 사용했지만, 입력 데이터의 구간이 달라졌기 때문에 결과 숫자가 다르다 (예_strid...)

**Q4. 이미지 분류 구조에서는 이렇게 추출·요약한 특징을 어떤 단계로 연결해 클래스별 점수를 만드나요?**

- 답변:
- 합성곱Convolution은 필터를 사용해서 특징(작은구역) 추출
- ReLU는 음수를 0으로 바꾸는 비선형 변환
- Max Pooling은 주변 특징을 대표값으로 요약(구간별 최댓값으로 요약하면서 크기가 작아진다)
- Flatten은 특징을 한 줄로 펼침(1차원 배열로 펼쳐진다)
- Dense는 특징들을 결합해서 클래스별로 점수 계산(가중치, 편향을 적용해서 최종 점수 계산)
- 이번 과제는 Max Pooling까지만 계산했고 실제로 이미지 분류를 구현하거나 학습하지는 않았다.


## 제출 전 확인

- 실습 1과 실습 2, 기초 과제의 요구사항을 모두 확인했나요?
- 코드 출력과 손계산을 비교했나요?
- 문제마다 Q1~Q4에 답했나요?
- 기온·특징 맵·클래스 점수의 의미를 구분해서 설명했나요?


---

## 실습 마무리

실습 1·실습 2·기초 과제에서 다룬 내용을 자신의 말로 정리하세요. 새로운 심화 문제가 아니라 **배운 개념과 계산 과정을 확인하는 복습 질문**입니다.

1. Convolution, Filter, Feature Map은 각각 무엇인가요? 이번 기온 실습의 예를 하나씩 들어 설명하세요.

2. 입력 길이 8, 필터 길이 3, stride 1, padding 없음일 때 출력 길이는 왜 6인가요? 두 필터를 적용하면 무엇이 두 개 생기나요?

3. 차이 필터 [−1, 0, 1]의 결과가 양수라는 것은 무엇을 뜻하나요? ReLU에서 0이 되었다면 원래 차이를 정확히 알 수 있나요?

4. [0.24, 0.26]에 Max Pooling과 Average Pooling을 적용한 결과를 각각 계산하세요. 풀링이 남기는 정보와 잃는 정보도 설명하세요.

5. 이미지 분류 예시의 처리 순서와 출력 크기를 적으세요. 입력은 8×8×1, 합성곱은 3×3 필터 2개, 풀링은 2×2이고, 나머지 설정은 문제 2-2와 같습니다. Flatten과 Dense는 어떻게 다른가요?

6. 이번 실습에서 이미지 분류 모델을 학습했나요? 기온 실습과 이미지 CNN의 공통점·차이점, 고정 필터와 학습되는 필터의 차이를 설명하세요.


### 실습 마무리 답변 작성란

**1. Convolution, Filter, Feature Map은 각각 무엇인가요? 이번 기온 실습의 예를 하나씩 들어 설명하세요.**

- 답변: Convolution은 작은 구간의 값과 필터 값을 같은 위치끼리 곱한 뒤 더하는 계산을 의미
- Filter는 그 계산에 사용할 가중치 묶음을 의미
- Feature Map는 같은 필터를 여러 위치로 옮겨가며 적용한 결과의 배열을 뜻한다.


**2. 입력 길이 8, 필터 길이 3, stride 1, padding 없음일 때 출력 길이는 왜 6인가요? 두 필터를 적용하면 무엇이 두 개 생기나요?**

- 답변: 필터 시작 위치가 0~5의 여섯 곳이기 때문에 8-3+1 = 6
- 두 필터를 각각 같은 입력에 적용하면 길이 6의 특징 맵 두개가 생긴다.
- 3개씩 묶기 때문에 6번, 7번은 3개의 계산이 되지 않기 때문에 8-3+1 = 6개가 됨


**3. 차이 필터 [−1, 0, 1]의 결과가 양수라는 것은 무엇을 뜻하나요? ReLU에서 0이 되었다면 원래 차이를 정확히 알 수 있나요?**

- 답변: 양수값이 나왔다면 마지막 기온이 처음 기온보다 높다라는 뜻
- ReLU 출력이 0이면 원래 차이가 음수였는지, 0이었는지, 음수였다면 얼마나 작았는지 등을 알기 어렵다.



**4. [0.24, 0.26]에 Max Pooling과 Average Pooling을 적용한 결과를 각각 계산하세요. 풀링이 남기는 정보와 잃는 정보도 설명하세요.**

- 답변: Max는 0.26 / Average는 (0.24+0.26) / 2 = 0.25
- [0.10, 0.26]에도 0.26의 정보를 남기기 때문에 플링은 계산할 값 수를 줄이는 대신 정보를 일부 잃는다.
- (플링 값을 사용하면 계산할 값 정보의 일부를 잃을 수 있다)



**5. 이미지 분류 예시의 처리 순서와 출력 크기를 적으세요. 입력은 8×8×1, 합성곱은 3×3 필터 2개, 풀링은 2×2이고, 나머지 설정은 문제 2-2와 같습니다. Flatten과 Dense는 어떻게 다른가요?**

- 답변: 입력층 8x8x1 -> convolution 6x6x2 -> ReLU 6x6x2 -> Max Pooling 3x3x2 -> Flatten 18 -> Dense 3 


**6. 이번 실습에서 이미지 분류 모델을 학습했나요? 기온 실습과 이미지 CNN의 공통점·차이점, 고정 필터와 학습되는 필터의 차이를 설명하세요.**

- 답변: 이번 실습에서는 이미지 분류 모델을 수행하지 않음. 기온 데이터를 1차원 합성곱, ReLU, 플링을 사용하여 계산하고 이미지 분류는 순서와 구조확인함.
